# YeatGPT train

**Content warning:** explicit language. This is a character simulation, not the real artist.

Do not train on yeat-only data. `data/processed/sft.jsonl` has to already mix general instruct rows. If that file is missing, stop. Do not build a lyrics-only set on this runtime.

Use an L4 or A100 (16GB or more) for Qwen2.5-7B. A free T4 falls back to Qwen3-4B-Instruct-2507. Leave this notebook closed until `sft.jsonl` exists so paid units are not spent on a broken run.

Push the repo first, then clone it here and upload `sft.jsonl` from your machine.

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

REPO = Path('/content/yeat-gpt')
if not (REPO / 'pyproject.toml').exists():
    subprocess.check_call([
        'git', 'clone', 'https://github.com/claytomode/yeat-gpt.git', str(REPO)
    ])
os.chdir(REPO)
sys.path.insert(0, str(REPO / 'src'))
print(REPO)

In [ ]:
from google.colab import files
from pathlib import Path

dest = Path('data/processed')
dest.mkdir(parents=True, exist_ok=True)
uploaded = files.upload()
if 'sft.jsonl' not in uploaded:
    raise SystemExit('Upload sft.jsonl. Not training without the mixed set.')
target = dest / 'sft.jsonl'
target.write_bytes(uploaded['sft.jsonl'])
print(f'Wrote {target} ({target.stat().st_size} bytes)')

In [ ]:
!pip install -q unsloth pyyaml

In [ ]:
from pathlib import Path

from yeat_gpt.train.sft import describe_choice, gpu_report, run_train, select_config

sft = Path('data/processed/sft.jsonl')
if not sft.is_file():
    raise SystemExit(
        'Missing data/processed/sft.jsonl. Upload the mixed set before training. '
        'Do not train on yeat-only data.'
    )
name, vram = gpu_report()
config = select_config(vram)
print(f'GPU: {name} ({vram:.1f} GB)')
print(describe_choice(vram))
print('The next cell prints the first formatted example, then trains.')
run_train(config_path=config, output_dir=Path('artifacts/yeat-gpt-lora'), gguf=False)

In [ ]:
# Optional q4_k_m GGUF. Off by default. Uncomment after the adapter exists.
# from pathlib import Path
# from yeat_gpt.train.sft import run_train, select_config, gpu_report
# _name, vram = gpu_report()
# run_train(config_path=select_config(vram), output_dir=Path('artifacts/yeat-gpt-lora'), gguf=True)